# 🗽 New York City Crime Intelligence — Pipeline ML Complet v2

## Contexte Business
> *"Le NYPD traite +200 000 arrestations par an. Notre système prédit la sévérité d'un crime et le profil probable du criminel pour optimiser l'allocation des ressources policières."*

## Pourquoi NYC ?
- **Données démographiques** incluses (âge, genre, race) → profiling natif impossible sur Chicago
- **Felony/Misdemeanor** = cible de sévérité claire et équilibrée (39% / 59%)
- **5 boroughs** = structure géographique naturelle pour K-Means
- **77 commissariats** (precinct) = précision géographique bien supérieure au borough seul

## Pipeline — 5 modèles enchaînés
```
Step 0 — K-Means           : Zone géographique (5 boroughs)
Step 1 — Random Forest     : Type de crime (4 groupes)
Step 2 — XGBoost optimisé  : Sévérité (Felony / Misdemeanor / Violation)
Step 3A — Random Forest    : Tranche d'âge du criminel
Step 3B — Random Forest    : Genre du criminel
Step 4 — Random Forest     : Prédiction temporelle (quel crime demain ?)
```

## Dataset
| | |
|---|---|
| **Source** | NYC Open Data — NYPD Arrest Data |
| **URL current** | https://data.cityofnewyork.us/resource/uip8-fykc.json |
| **URL historic** | https://data.cityofnewyork.us/resource/8h9b-rp9u.json |
| **Colonnes chargées** | 14 (toutes les colonnes utiles ML) |
| **Volume** | ~200 000 arrestations |


## ⚙️ 0 — Imports

In [ ]:
# !pip install pandas numpy matplotlib seaborn scikit-learn xgboost joblib requests --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import requests, warnings, joblib, os
from io import StringIO

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_score, RandomizedSearchCV
)
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix
)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted', font_scale=1.05)
plt.rcParams.update({'figure.dpi':120, 'figure.facecolor':'white'})
RANDOM_STATE = 42
print('✅ Imports OK')


## 📥 1 — Chargement des Données NYC (14 colonnes)

In [ ]:
# NYC NYPD Arrests — toutes les colonnes utiles ML
print('⏳ Chargement NYC NYPD Arrests...')

URL_CURRENT  = 'https://data.cityofnewyork.us/resource/uip8-fykc.json'
URL_HISTORIC = 'https://data.cityofnewyork.us/resource/8h9b-rp9u.json'

# 14 colonnes — toutes celles utiles pour le ML
# Exclues volontairement : arrest_key (ID), x_coord_cd/y_coord_cd (doublon GPS),
#                          geocoded_column (doublon), pd_cd/ky_cd (redondants avec ofns_desc)
PARAMS = {
    '$limit': 200000,
    '$order': 'arrest_date DESC',
    '$select': ','.join([
        'arrest_date',        # date → heure, jour, mois
        'ofns_desc',          # type de crime (catégorie générale)
        'pd_desc',            # description détaillée
        'law_cat_cd',         # sévérité : F/M/V ← CIBLE STEP 2
        'law_code',           # code légal exact (ex: PL 121.12)
        'arrest_boro',        # borough : M/K/B/Q/S
        'arrest_precinct',    # commissariat (1-77) ← nouvelle feature puissante
        'jurisdiction_code',  # type de police : 0=NYPD, 1=Transit, 2=Housing
        'age_group',          # tranche d'âge ← CIBLE STEP 3A
        'perp_sex',           # genre ← CIBLE STEP 3B
        'perp_race',          # origine ethnique
        'latitude',           # GPS
        'longitude',          # GPS
    ])
}

try:
    r = requests.get(URL_CURRENT, params=PARAMS, timeout=120)
    r.raise_for_status()
    df_raw = pd.DataFrame(r.json())
    print(f'✅ {len(df_raw):,} arrestations (année en cours)')
except Exception as e:
    print(f'⚠️ {e} — tentative historique 2022→2026...')
    PARAMS['$where'] = "arrest_date >= '2022-01-01'"
    r = requests.get(URL_HISTORIC, params=PARAMS, timeout=120)
    df_raw = pd.DataFrame(r.json())
    print(f'✅ {len(df_raw):,} arrestations (historique 2022→2026)')

print(f'\nColonnes chargées : {df_raw.columns.tolist()}')
print(f'Shape : {df_raw.shape}')
df_raw.head(3)


## 🔍 2 — EDA

In [ ]:
print('='*60)
print('  STATISTIQUES GÉNÉRALES — NYC NYPD ARRESTS')
print('='*60)
print(f'  Lignes          : {len(df_raw):,}')
print(f'  Colonnes        : {df_raw.shape[1]}')
print(f'\n  Sévérité (loi_cat_cd) :')
for v, n in df_raw['law_cat_cd'].value_counts().items():
    pct = n/len(df_raw)*100
    print(f'    {v:<3} → {n:>6,} ({pct:>5.1f}%)')
print(f'\n  Boroughs :')
boro_names = {'M':'Manhattan','K':'Brooklyn','B':'Bronx','Q':'Queens','S':'Staten Island'}
for v, n in df_raw['arrest_boro'].value_counts().items():
    print(f'    {boro_names.get(v,v):<15} → {n:>6,} ({n/len(df_raw)*100:.1f}%)')
print(f'\n  Tranches d\'âge :')
print(df_raw['age_group'].value_counts().head(7))
print(f'\n  Genre :')
print(df_raw['perp_sex'].value_counts())
print(f'\n  Top 5 précinct :')
print(df_raw['arrest_precinct'].value_counts().head(5))
print(f'\n  Juridictions :')
jur = {'0':'NYPD','1':'Transit Police','2':'Housing Police'}
for v, n in df_raw['jurisdiction_code'].value_counts().items():
    print(f'    {jur.get(str(v),str(v)):<20} → {n:>6,}')


In [ ]:
# ── VIZ 1 : Vue d'ensemble ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top 15 crimes
top15 = df_raw['ofns_desc'].value_counts().head(15)
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, 15))
axes[0,0].barh(top15.index[::-1], top15.values[::-1], color=colors, edgecolor='white', lw=0.4)
axes[0,0].set_title('Top 15 Types de Crimes NYC', fontweight='bold')
axes[0,0].set_xlabel("Nombre d'arrestations")
axes[0,0].spines['top'].set_visible(False); axes[0,0].spines['right'].set_visible(False)

# Sévérité
sev_map = {'F':'Felony','M':'Misdemeanor','V':'Violation','I':'Infraction'}
sev_counts = df_raw['law_cat_cd'].value_counts().head(4)
axes[0,1].pie(sev_counts.values,
              labels=[sev_map.get(k,k) for k in sev_counts.index],
              colors=['#ef4444','#f59e0b','#10b981','#6366f1'],
              autopct='%1.1f%%', startangle=90,
              wedgeprops={'edgecolor':'white','linewidth':2})
axes[0,1].set_title('Répartition Sévérité', fontweight='bold')

# Âge
age_order = ['<18','18-24','25-44','45-64','65+']
age_counts = df_raw['age_group'].value_counts()
age_counts = age_counts.reindex([a for a in age_order if a in age_counts.index], fill_value=0)
axes[1,0].bar(age_counts.index, age_counts.values,
              color=['#3b82f6','#ef4444','#10b981','#f59e0b','#8b5cf6'][:len(age_counts)],
              edgecolor='white')
axes[1,0].set_title("Répartition par Tranche d'âge", fontweight='bold')
axes[1,0].set_ylabel("Nombre d'arrestations")
axes[1,0].spines['top'].set_visible(False); axes[1,0].spines['right'].set_visible(False)

# Top precincts
top_prec = df_raw['arrest_precinct'].value_counts().head(10)
axes[1,1].barh(top_prec.index[::-1].astype(str), top_prec.values[::-1],
               color='#6366f1', edgecolor='white', alpha=0.85)
axes[1,1].set_title('Top 10 Commissariats (Precinct)', fontweight='bold')
axes[1,1].set_xlabel("Nombre d'arrestations")
axes[1,1].spines['top'].set_visible(False); axes[1,1].spines['right'].set_visible(False)

plt.suptitle("🗽 Vue d'ensemble — NYC NYPD Arrests", fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('viz_01_overview.png', bbox_inches='tight')
plt.show()
print('✅ viz_01_overview.png')


In [ ]:
# ── VIZ 2 : Patterns temporels ──
df_raw['date_p']     = pd.to_datetime(df_raw['arrest_date'], errors='coerce')
df_raw['hour']       = df_raw['date_p'].dt.hour.fillna(12).astype(int)
df_raw['day_of_week']= df_raw['date_p'].dt.dayofweek.fillna(0).astype(int)
df_raw['month']      = df_raw['date_p'].dt.month.fillna(6).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Heure × sévérité
hour_sev = (df_raw[df_raw['law_cat_cd'].isin(['F','M'])]
            .groupby(['hour','law_cat_cd']).size()
            .unstack(fill_value=0)
            .reindex(range(24), fill_value=0))
if 'F' in hour_sev.columns:
    axes[0].plot(range(24), hour_sev['F'], 'o-', color='#ef4444', lw=2, ms=4, label='Felony')
if 'M' in hour_sev.columns:
    axes[0].plot(range(24), hour_sev['M'], 's-', color='#f59e0b', lw=2, ms=4, label='Misdemeanor')
axes[0].set_title('Arrestations par Heure', fontweight='bold')
axes[0].set_xlabel('Heure'); axes[0].legend()
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

# Jour
days_fr = ['Lun','Mar','Mer','Jeu','Ven','Sam','Dim']
day_counts = df_raw['day_of_week'].value_counts().reindex(range(7), fill_value=0)
axes[1].bar(range(7), day_counts.values, color='#3b82f6', alpha=0.8, edgecolor='white')
axes[1].set_xticks(range(7)); axes[1].set_xticklabels(days_fr)
axes[1].set_title('Volume par Jour', fontweight='bold')
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

# Borough × Felony rate
boro_felony = df_raw.groupby('arrest_boro').apply(
    lambda x: (x['law_cat_cd']=='F').mean()*100).sort_values(ascending=False)
boro_felony.index = [boro_names.get(k,k) for k in boro_felony.index]
axes[2].bar(boro_felony.index, boro_felony.values,
            color=['#ef4444','#f59e0b','#10b981','#3b82f6','#8b5cf6'][:len(boro_felony)],
            edgecolor='white')
axes[2].axhline(y=df_raw[df_raw['law_cat_cd']=='F'].shape[0]/len(df_raw)*100,
                color='black', linestyle='--', lw=1.5, label='Moy.')
axes[2].set_title('Taux Felony par Borough (%)', fontweight='bold')
axes[2].tick_params(axis='x', rotation=15); axes[2].legend()
axes[2].spines['top'].set_visible(False); axes[2].spines['right'].set_visible(False)

plt.suptitle('⏰ Patterns Temporels & Géographiques', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('viz_02_temporal.png', bbox_inches='tight')
plt.show()


## 🧹 3 — Feature Engineering (14 colonnes)

In [ ]:
df = df_raw.copy()

# ── Coordonnées GPS ──
df['latitude']  = pd.to_numeric(df['latitude'],  errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
df = df.dropna(subset=['latitude','longitude','ofns_desc','law_cat_cd'])
df = df[(df['latitude'] != 0) & (df['longitude'] != 0)]

# ── Temporel ──
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

# ── Borough → numérique ──
boro_num = {'M':0,'K':1,'B':2,'Q':3,'S':4}
df['location_group'] = df['arrest_boro'].map(boro_num).fillna(0).astype(int)

# ── NOUVELLE : Precinct (commissariat) ──
# C'est la feature la plus puissante géographiquement (77 valeurs vs 5 boroughs)
df['precinct'] = pd.to_numeric(df['arrest_precinct'], errors='coerce').fillna(0).astype(int)

# ── NOUVELLE : Jurisdiction (type de police) ──
# 0=NYPD standard, 1=Transit (métro), 2=Housing (logements sociaux)
df['jurisdiction'] = pd.to_numeric(df['jurisdiction_code'], errors='coerce').fillna(0).astype(int)

# ── NOUVELLE : Law code prefix (type de loi) ──
# PL=Penal Law, VTL=Vehicle Traffic, AL=Alcohol, etc.
def extract_law_prefix(code):
    if pd.isna(code) or str(code).strip() == '': return 0
    code = str(code).strip().upper()
    prefixes = {'PL':1,'VTL':2,'AL':3,'RL':4,'GML':5,'TPL':6,'ECL':7}
    for p, v in prefixes.items():
        if code.startswith(p): return v
    return 8  # autres
df['law_prefix'] = df['law_code'].apply(extract_law_prefix)

# ── Cible 1 : Type de crime (4 groupes) ──
def group_crime(crime):
    if pd.isna(crime): return 'PUBLIC ORDER'
    c = str(crime).upper()
    if any(w in c for w in ['ASSAULT','ROBBERY','WEAPON','HOMICIDE','RAPE','SEX CRIME','MURDER','KIDNAP']): return 'VIOLENT'
    if any(w in c for w in ['THEFT','LARCENY','BURGLARY','VEHICLE','FRAUD','FORGERY','STOLEN']): return 'PROPERTY'
    if any(w in c for w in ['DRUG','NARCOTIC','CONTROLLED','MARIJUANA','CANNABIS']): return 'DRUG'
    return 'PUBLIC ORDER'

df['crime_group']   = df['ofns_desc'].apply(group_crime)
le_group            = LabelEncoder()
df['crime_encoded'] = le_group.fit_transform(df['crime_group'])

# ── Cible 2 : Sévérité ──
sev_map_lbl = {'F':'FELONY','M':'MISDEMEANOR','V':'VIOLATION','I':'VIOLATION','9':'MISDEMEANOR'}
df['severity']    = df['law_cat_cd'].map(sev_map_lbl).fillna('MISDEMEANOR')
le_sev            = LabelEncoder()
df['sev_encoded'] = le_sev.fit_transform(df['severity'])

# ── Cibles 3 : Profiling ──
valid_ages          = ['<18','18-24','25-44','45-64','65+']
df['age_clean']     = df['age_group'].apply(lambda x: x if x in valid_ages else None)
df                  = df.dropna(subset=['age_clean'])
le_age              = LabelEncoder()
df['age_encoded']   = le_age.fit_transform(df['age_clean'])

df['gender_clean']  = df['perp_sex'].map({'M':0,'F':1})
df                  = df.dropna(subset=['gender_clean'])
df['gender_clean']  = df['gender_clean'].astype(int)

# Race encoding
le_race             = LabelEncoder()
df['race_clean']    = df['perp_race'].fillna('UNKNOWN')
df['race_encoded']  = le_race.fit_transform(df['race_clean'])

print(f'✅ Dataset final : {len(df):,} lignes')
print(f'\n📊 Nouvelles features créées :')
print(f'   precinct      : {df["precinct"].nunique()} commissariats distincts')
print(f'   jurisdiction  : {df["jurisdiction"].value_counts().to_dict()}')
print(f'   law_prefix    : {df["law_prefix"].value_counts().head(4).to_dict()}')
print(f'\n🎯 Cibles :')
print(f'   crime_group   : {df["crime_group"].value_counts().to_dict()}')
print(f'   severity      : {df["severity"].value_counts().to_dict()}')
print(f'   age_clean     : {df["age_clean"].value_counts().to_dict()}')
print(f'   gender        : Homme={( df["gender_clean"]==0).sum():,} | Femme={(df["gender_clean"]==1).sum():,}')


## 🗺️ 4 — K-Means : Zones Géographiques

In [ ]:
coords = df[['latitude','longitude']].sample(min(60000,len(df)), random_state=RANDOM_STATE)
scaler_kmeans = StandardScaler()
X_geo = scaler_kmeans.fit_transform(coords)

# Elbow
inertias = []
for k in range(2,9):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=5)
    inertias.append(km.fit(X_geo).inertia_)

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(range(2,9), inertias, 'o-', color='#3b82f6', lw=2.5, ms=8)
ax.axvline(x=5, color='#ef4444', linestyle='--', lw=1.8, label='k=5 = 5 boroughs NYC')
ax.set_xlabel('k'); ax.set_ylabel('Inertie')
ax.set_title('Méthode Elbow — k=5 correspond naturellement aux 5 boroughs', fontweight='bold')
ax.legend(); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('viz_03_elbow.png', bbox_inches='tight')
plt.show()

kmeans = KMeans(n_clusters=5, random_state=RANDOM_STATE, n_init=10)
kmeans.fit(X_geo)
coords_c = coords.copy(); coords_c['cluster'] = kmeans.labels_
df = df.merge(coords_c[['cluster']], left_index=True, right_index=True, how='left')
df['cluster'] = df['cluster'].fillna(0).astype(int)
print(f'✅ K-Means : 5 clusters')
print(df['cluster'].value_counts().sort_index())


In [ ]:
# ── VIZ : Clusters ──
BORO_COLORS = ['#3b82f6','#ef4444','#f59e0b','#10b981','#8b5cf6']
BORO_LABELS = ['Manhattan','Brooklyn','Bronx','Queens','Staten Island']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sample = df.sample(min(12000,len(df)), random_state=42)
for c in range(5):
    m = sample['cluster']==c
    axes[0].scatter(sample.loc[m,'longitude'], sample.loc[m,'latitude'],
                    c=BORO_COLORS[c], s=3, alpha=0.4, label=BORO_LABELS[c])
cents = scaler_kmeans.inverse_transform(kmeans.cluster_centers_)
axes[0].scatter(cents[:,1], cents[:,0], c='black', s=180, marker='X', zorder=10, label='Centroïdes')
axes[0].set_title('K-Means NYC — Les 5 clusters = 5 boroughs ✅', fontweight='bold')
axes[0].legend(fontsize=9, markerscale=3)

felony_cl = df.groupby('cluster').apply(lambda x: (x['severity']=='FELONY').mean()*100)
axes[1].bar([BORO_LABELS[i] for i in range(5)], felony_cl.values,
            color=BORO_COLORS, edgecolor='white')
axes[1].axhline(y=(df['severity']=='FELONY').mean()*100, color='black',
                linestyle='--', lw=1.5,
                label=f"Moy. {(df['severity']=='FELONY').mean()*100:.1f}%")
axes[1].set_title('Taux Felony par Cluster', fontweight='bold')
axes[1].set_ylabel('% Felony'); axes[1].legend()
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.suptitle('📍 K-Means NYC — Validation géographique', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('viz_04_kmeans.png', bbox_inches='tight')
plt.show()


## 🤖 5 — Modèles Supervisés

### Features par modèle — justification des ajouts

| Feature | Step1 RF | Step2 XGB | Step3 RF | Pourquoi |
|---|---|---|---|---|
| latitude/longitude | ✅ | ✅ | ✅ | Localisation précise |
| cluster | ✅ | ✅ | ✅ | Zone K-Means |
| **precinct** | ✅ | ✅ | ✅ | 77 commissariats > 5 boroughs |
| **jurisdiction** | ✅ | ✅ | ❌ | Transit/Housing vs NYPD |
| hour/day/month | ✅ | ✅ | ✅ | Temporel |
| **law_prefix** | ❌ | ✅ | ❌ | Type de loi → sévérité |
| crime_encoded | ❌ | ✅ | ✅ | Type de crime prédit |
| sev_encoded | ❌ | ❌ | ✅ | Sévérité prédite |


In [ ]:
# ── Définition des features ──

# Step 1 : RF Crime Type
# + precinct et jurisdiction par rapport à la v1
FEAT_S1 = [
    'latitude','longitude','hour','day_of_week','month','is_weekend',
    'cluster','location_group',
    'precinct',      # NOUVEAU — commissariat (77 valeurs)
    'jurisdiction',  # NOUVEAU — type de police
]

# Step 2 : XGBoost Sévérité
# + law_prefix par rapport à la v1
FEAT_S2 = FEAT_S1 + [
    'crime_encoded',  # type de crime prédit Step 1
    'law_prefix',     # NOUVEAU — type de loi (PL, VTL, AL...)
]

# Step 3 : Profiling (âge + genre)
FEAT_PROF = FEAT_S2 + ['sev_encoded']

# Step 4 : Temporel (sans lat/lon car on n'a pas les coordonnées à l'avance)
FEAT_TEMP = [
    'hour','day_of_week','month','is_weekend',
    'cluster','location_group','jurisdiction',
]

print('✅ Features définies')
print(f'  Step 1 ({len(FEAT_S1)} features) : {FEAT_S1}')
print(f'  Step 2 ({len(FEAT_S2)} features) : {FEAT_S2}')
print(f'  Step 3 ({len(FEAT_PROF)} features) : {FEAT_PROF}')
print(f'  Step 4 ({len(FEAT_TEMP)} features) : {FEAT_TEMP}')


In [ ]:
# ── STEP 1 : Random Forest — Type de crime ──
df_s1 = df[FEAT_S1+['crime_encoded']].dropna()
X1_tr,X1_te,y1_tr,y1_te = train_test_split(
    df_s1[FEAT_S1], df_s1['crime_encoded'],
    test_size=.2, random_state=RANDOM_STATE, stratify=df_s1['crime_encoded'])

print('⏳ Step 1 — Random Forest Crime Type...')
rf_crime = RandomForestClassifier(
    n_estimators=200, max_depth=12,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
rf_crime.fit(X1_tr, y1_tr)
y1_pred = rf_crime.predict(X1_te)
acc_s1 = accuracy_score(y1_te, y1_pred)
f1_s1  = f1_score(y1_te, y1_pred, average='weighted')
print(f'   ✅ Accuracy: {acc_s1*100:.1f}%  F1: {f1_s1*100:.1f}%')
print(classification_report(y1_te, y1_pred, target_names=le_group.classes_))
df['crime_predicted'] = rf_crime.predict(df[FEAT_S1].fillna(0))


In [ ]:
# ── STEP 2 : XGBoost + RandomizedSearchCV — Sévérité ──
df_s2 = df[FEAT_S2+['sev_encoded']].dropna()
X2_tr,X2_te,y2_tr,y2_te = train_test_split(
    df_s2[FEAT_S2], df_s2['sev_encoded'],
    test_size=.2, random_state=RANDOM_STATE, stratify=df_s2['sev_encoded'])

print('⏳ Step 2 — RandomizedSearchCV + XGBoost Sévérité...')
param_dist = {
    'n_estimators':  [200,300,400],
    'max_depth':     [4,5,6,7],
    'learning_rate': [0.05,0.08,0.1],
    'subsample':     [0.8,0.9,1.0],
    'colsample_bytree': [0.8,0.9,1.0],
}
xgb_base = XGBClassifier(
    random_state=RANDOM_STATE, eval_metric='mlogloss',
    verbosity=0, n_jobs=-1)
search = RandomizedSearchCV(
    xgb_base, param_dist, n_iter=15,
    scoring='f1_weighted', cv=3,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1)
sample_idx = X2_tr.sample(min(25000,len(X2_tr)), random_state=RANDOM_STATE).index
search.fit(X2_tr.loc[sample_idx], y2_tr.loc[sample_idx])
print(f'   Meilleurs params : {search.best_params_}')

xgb_sev = XGBClassifier(
    **search.best_params_,
    random_state=RANDOM_STATE, eval_metric='mlogloss',
    verbosity=0, n_jobs=-1)
xgb_sev.fit(X2_tr, y2_tr)
y2_pred = xgb_sev.predict(X2_te)
acc_s2  = accuracy_score(y2_te, y2_pred)
f1_s2   = f1_score(y2_te, y2_pred, average='weighted')
auc_s2  = roc_auc_score(y2_te, xgb_sev.predict_proba(X2_te), multi_class='ovr')
print(f'   ✅ Accuracy: {acc_s2*100:.1f}%  F1: {f1_s2*100:.1f}%  AUC: {auc_s2:.3f}')
print(classification_report(y2_te, y2_pred, target_names=le_sev.classes_))
df['sev_predicted'] = xgb_sev.predict(df[FEAT_S2].fillna(0))


In [ ]:
# ── STEP 3A : Random Forest — Âge ──
df_age = df[FEAT_PROF+['age_encoded']].dropna()
X_a_tr,X_a_te,y_a_tr,y_a_te = train_test_split(
    df_age[FEAT_PROF], df_age['age_encoded'],
    test_size=.2, random_state=RANDOM_STATE, stratify=df_age['age_encoded'])

print('⏳ Step 3A — RF Âge...')
rf_age = RandomForestClassifier(
    n_estimators=200, max_depth=10,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
rf_age.fit(X_a_tr, y_a_tr)
y_a_pred = rf_age.predict(X_a_te)
acc_age  = accuracy_score(y_a_te, y_a_pred)
f1_age   = f1_score(y_a_te, y_a_pred, average='weighted')
print(f'   ✅ Accuracy: {acc_age*100:.1f}%  F1: {f1_age*100:.1f}%')

# ── STEP 3B : Random Forest — Genre ──
df_gen = df[FEAT_PROF+['gender_clean']].dropna()
X_g_tr,X_g_te,y_g_tr,y_g_te = train_test_split(
    df_gen[FEAT_PROF], df_gen['gender_clean'],
    test_size=.2, random_state=RANDOM_STATE, stratify=df_gen['gender_clean'])

print('⏳ Step 3B — RF Genre...')
rf_gender = RandomForestClassifier(
    n_estimators=200, max_depth=10,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
rf_gender.fit(X_g_tr, y_g_tr)
y_g_pred  = rf_gender.predict(X_g_te)
y_g_prob  = rf_gender.predict_proba(X_g_te)[:,1]
acc_gen   = accuracy_score(y_g_te, y_g_pred)
f1_gen    = f1_score(y_g_te, y_g_pred, average='weighted')
auc_gen   = roc_auc_score(y_g_te, y_g_prob)
print(f'   ✅ Accuracy: {acc_gen*100:.1f}%  F1: {f1_gen*100:.1f}%  AUC: {auc_gen:.3f}')


In [ ]:
# ── STEP 4 : Random Forest — Prédiction temporelle ──
df_temp = df[FEAT_TEMP+['crime_encoded']].dropna()
X_t_tr,X_t_te,y_t_tr,y_t_te = train_test_split(
    df_temp[FEAT_TEMP], df_temp['crime_encoded'],
    test_size=.2, random_state=RANDOM_STATE, stratify=df_temp['crime_encoded'])

print('⏳ Step 4 — RF Temporel...')
rf_temporal = RandomForestClassifier(
    n_estimators=200, max_depth=10,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
rf_temporal.fit(X_t_tr, y_t_tr)
y_t_pred = rf_temporal.predict(X_t_te)
acc_temp = accuracy_score(y_t_te, y_t_pred)
f1_temp  = f1_score(y_t_te, y_t_pred, average='weighted')
print(f'   ✅ Accuracy: {acc_temp*100:.1f}%  F1: {f1_temp*100:.1f}%')


In [ ]:
# ── VIZ : Résumé performances ──
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

steps  = ['Step 1\nCrime Type\n(RF)', 'Step 2\nSévérité\n(XGB)',
          'Step 3A\nÂge\n(RF)', 'Step 3B\nGenre\n(RF)']
accs   = [acc_s1*100, acc_s2*100, acc_age*100, acc_gen*100]
f1s    = [f1_s1*100,  f1_s2*100,  f1_age*100,  f1_gen*100]
colors = ['#3b82f6','#8b5cf6','#f59e0b','#ef4444']
x = np.arange(len(steps)); w = 0.35

bars1 = axes[0].bar(x-w/2, accs, w, color=colors, alpha=0.9, edgecolor='white', label='Accuracy')
bars2 = axes[0].bar(x+w/2, f1s,  w, color=colors, alpha=0.5, edgecolor='white', label='F1', hatch='//')
for b,v in zip(bars1,accs): axes[0].text(b.get_x()+b.get_width()/2, v+.5, f'{v:.1f}%', ha='center', fontsize=9, fontweight='bold')
for b,v in zip(bars2,f1s):  axes[0].text(b.get_x()+b.get_width()/2, v+.5, f'{v:.1f}%', ha='center', fontsize=9)
axes[0].set_xticks(x); axes[0].set_xticklabels(steps, fontsize=10)
axes[0].set_ylim(0,110); axes[0].set_ylabel('Score (%)')
axes[0].set_title('Performance du Pipeline', fontweight='bold')
axes[0].legend(); axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

# Feature importances XGBoost
fi = pd.Series(xgb_sev.feature_importances_, index=FEAT_S2).sort_values()
col_fi = ['#ef4444' if v > fi.median() else '#e2e8f0' for v in fi.values]
axes[1].barh(fi.index, fi.values*100, color=col_fi, edgecolor='white')
axes[1].set_xlabel('Importance (%)'); axes[1].set_title('Feature Importances — XGBoost', fontweight='bold')
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.suptitle('🤖 Évaluation du Pipeline ML', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('viz_05_performance.png', bbox_inches='tight')
plt.show()
print('✅ viz_05_performance.png')


## 💰 6 — Analyse Coût/Gain

In [ ]:
# Coûts réels NYPD (basés sur le budget public)
COST  = {'FELONY':1200, 'MISDEMEANOR':350, 'VIOLATION':80}
WASTE = 500  # coût d'une mauvaise classification

true_labels = le_sev.inverse_transform(y2_te)
pred_labels = le_sev.inverse_transform(y2_pred)

correct = sum(t==p for t,p in zip(true_labels,pred_labels))
wrong   = len(true_labels) - correct
savings = sum(COST[t] for t,p in zip(true_labels,pred_labels) if t==p)
waste   = wrong * WASTE
net     = savings - waste
scale   = 200000 / len(true_labels)

print('='*60)
print('  ANALYSE COÛT/GAIN — NYC NYPD')
print('='*60)
print(f'\n  Sur {len(true_labels):,} prédictions :')
print(f'  ✅ Correctes : {correct:,} ({correct/len(true_labels)*100:.1f}%)')
print(f'  ❌ Erreurs   : {wrong:,} ({wrong/len(true_labels)*100:.1f}%)')
print(f'\n  Économies bonnes prédictions : ${savings:,.0f}')
print(f'  Coût des mauvaises           : ${waste:,.0f}')
print(f'  Gain net                     : ${net:,.0f}')
print(f'\n  📈 PROJECTION 200K arrestations/an :')
print(f'  💰 Économies estimées : ${net*scale:,.0f}/an')
print(f'  → ${net*scale/200000:.0f} économisés par arrestation traitée')


## 🔮 7 — Prédiction Temporelle

In [ ]:
from datetime import datetime, timedelta

def predict_scene(hour, day_name, month, borough, jurisdiction_val=0, top_n=4):
    day_map  = {'Lundi':0,'Mardi':1,'Mercredi':2,'Jeudi':3,'Vendredi':4,'Samedi':5,'Dimanche':6}
    boro_map = {'Manhattan':0,'Brooklyn':1,'Bronx':2,'Queens':3,'Staten Island':4}
    boro_coords = {
        'Manhattan':(40.7831,-73.9712),'Brooklyn':(40.6782,-73.9442),
        'Bronx':(40.8448,-73.8648),'Queens':(40.7282,-73.7949),
        'Staten Island':(40.5795,-74.1502),
    }
    day_num = day_map.get(day_name,0)
    cluster = boro_map.get(borough,0)
    is_we   = int(day_num>=5)

    xt = pd.DataFrame([dict(hour=hour,day_of_week=day_num,month=month,
                            is_weekend=is_we,cluster=cluster,
                            location_group=cluster,
                            jurisdiction=jurisdiction_val)])[FEAT_TEMP]
    crime_pr  = rf_temporal.predict_proba(xt)[0]
    top_idx   = crime_pr.argsort()[::-1][:top_n]
    top_crimes= [(le_group.inverse_transform([i])[0], round(float(crime_pr[i])*100,1))
                 for i in top_idx]
    crime_enc = int(top_idx[0])
    lat, lon  = boro_coords.get(borough,(40.7128,-74.0060))

    # Precinct médian du borough pour la prédiction
    boro_num_map = {'Manhattan':0,'Brooklyn':1,'Bronx':2,'Queens':3,'Staten Island':4}
    median_prec  = int(df[df['location_group']==boro_num_map.get(borough,0)]['precinct'].median())

    x2 = pd.DataFrame([dict(
        latitude=lat,longitude=lon,hour=hour,day_of_week=day_num,month=month,
        is_weekend=is_we,cluster=cluster,location_group=cluster,
        precinct=median_prec,jurisdiction=jurisdiction_val,
        crime_encoded=crime_enc,law_prefix=1)])[FEAT_S2]
    sev_enc  = int(xgb_sev.predict(x2)[0])
    sev_lbl  = le_sev.inverse_transform([sev_enc])[0]
    sev_conf = float(xgb_sev.predict_proba(x2)[0].max())

    return {'contexte':f'{borough} · {day_name} {hour}h',
            'top_crimes':top_crimes,'sévérité':sev_lbl,'confiance':f'{sev_conf*100:.1f}%'}

tomorrow = (datetime.now()+timedelta(days=1))
month    = tomorrow.month

print('='*60)
for boro,h,day in [('Bronx',23,'Vendredi'),('Manhattan',14,'Mercredi'),
                   ('Brooklyn',3,'Samedi'),('Queens',9,'Lundi')]:
    r = predict_scene(h, day, month, boro)
    print(f'\n📍 {r["contexte"]}')
    for c,p in r['top_crimes']:
        bar = '█'*int(p/4)
        print(f'   {c:<15} {p:>5.1f}%  {bar}')
    print(f'   Sévérité : {r["sévérité"]} ({r["confiance"]})')


## 🧬 8 — Profiling Criminel

In [ ]:
def profile_criminal(hour, day_name, month, borough, jurisdiction_val=0):
    day_map  = {'Lundi':0,'Mardi':1,'Mercredi':2,'Jeudi':3,'Vendredi':4,'Samedi':5,'Dimanche':6}
    boro_map = {'Manhattan':0,'Brooklyn':1,'Bronx':2,'Queens':3,'Staten Island':4}
    boro_coords = {'Manhattan':(40.7831,-73.9712),'Brooklyn':(40.6782,-73.9442),
                   'Bronx':(40.8448,-73.8648),'Queens':(40.7282,-73.7949),
                   'Staten Island':(40.5795,-74.1502)}
    day_num = day_map.get(day_name,0)
    cluster = boro_map.get(borough,0)
    is_we   = int(day_num>=5)
    lat,lon = boro_coords.get(borough,(40.7128,-74.0060))
    median_prec = int(df[df['location_group']==cluster]['precinct'].median())

    x1 = pd.DataFrame([dict(latitude=lat,longitude=lon,hour=hour,day_of_week=day_num,
                             month=month,is_weekend=is_we,cluster=cluster,
                             location_group=cluster,precinct=median_prec,
                             jurisdiction=jurisdiction_val)])[FEAT_S1]
    crime_enc = int(rf_crime.predict(x1)[0])
    crime_lbl = le_group.inverse_transform([crime_enc])[0]

    x2 = x1.copy(); x2['crime_encoded']=crime_enc; x2['law_prefix']=1
    x2 = x2[FEAT_S2]
    sev_enc  = int(xgb_sev.predict(x2)[0])
    sev_lbl  = le_sev.inverse_transform([sev_enc])[0]

    xp = x2.copy(); xp['sev_encoded']=sev_enc
    xp = xp[FEAT_PROF]
    age_pr   = rf_age.predict_proba(xp)[0]
    age_lbl  = le_age.inverse_transform([rf_age.predict(xp)[0]])[0]
    age_all  = {le_age.inverse_transform([i])[0]:round(float(p)*100,1) for i,p in enumerate(age_pr)}
    gen_enc  = int(rf_gender.predict(xp)[0])
    gen_lbl  = 'Femme' if gen_enc==1 else 'Homme'
    gen_conf = float(rf_gender.predict_proba(xp)[0].max())

    return {'borough':borough,'crime':crime_lbl,'sévérité':sev_lbl,
            'âge_prédit':age_lbl,'âge_probas':dict(sorted(age_all.items(),key=lambda x:-x[1])),
            'genre':gen_lbl,'genre_conf':f'{gen_conf*100:.1f}%'}

print('='*65)
print('  PROFILING CRIMINEL — NYC NYPD MODELS')
print('='*65)
for boro,h,day in [('Bronx',23,'Vendredi'),('Manhattan',14,'Mercredi'),('Brooklyn',3,'Samedi')]:
    r = profile_criminal(h,day,month,boro)
    print(f'\n📍 {r["borough"]} · {day} {h}h')
    print(f'   Crime : {r["crime"]}  |  Sévérité : {r["sévérité"]}')
    print(f'   Âge probable : {r["âge_prédit"]}')
    for a,p in r['âge_probas'].items():
        bar='█'*int(p/6)
        print(f'     {a:<8} {p:>5.1f}%  {bar}')
    print(f'   Genre : {r["genre"]} ({r["genre_conf"]})')


## 💡 9 — Insights

In [ ]:
print('='*65)
print('  INSIGHTS — NYC CRIME INTELLIGENCE')
print('='*65)

# Insight 1
print(f'\n⚖️ INSIGHT 1 — Distribution sévérité')
for s,n in df['severity'].value_counts().items():
    pct = n/len(df)*100
    bar = '█'*int(pct/3)
    print(f'   {s:<15} : {n:,} ({pct:.1f}%) {bar}')

# Insight 2
print(f'\n👤 INSIGHT 2 — Profil par type de crime')
for cg in le_group.classes_:
    sub = df[df['crime_group']==cg]
    if len(sub)==0: continue
    top_age = sub['age_clean'].mode()[0]
    pct_male= (sub['gender_clean']==0).mean()*100
    print(f'   {cg:<15} → {top_age} · Homme {pct_male:.0f}%')

# Insight 3
print(f'\n🏢 INSIGHT 3 — Commissariat le plus actif')
top_prec = df['precinct'].value_counts().head(3)
for prec, n in top_prec.items():
    pct_felony = (df[df['precinct']==prec]['severity']=='FELONY').mean()*100
    print(f'   Precinct {prec:<4} → {n:,} arrestations | {pct_felony:.0f}% Felony')

# Insight 4
print(f'\n⏰ INSIGHT 4 — Heure et sévérité')
night = df[df['hour'].isin(list(range(22,24))+list(range(0,6)))]
jour  = df[df['hour'].between(10,16)]
print(f'   Nuit (22h-6h)  : {(night["severity"]=="FELONY").mean()*100:.1f}% Felony')
print(f'   Jour (10h-16h) : {(jour["severity"]=="FELONY").mean()*100:.1f}% Felony')

# Résumé
print(f'\n🤖 RÉSUMÉ PIPELINE :')
print(f'   K-Means  Step 0  : 5 zones = 5 boroughs NYC')
print(f'   RF       Step 1  : {acc_s1*100:.1f}% — type de crime ({len(FEAT_S1)} features)')
print(f'   XGBoost  Step 2  : {acc_s2*100:.1f}% — sévérité AUC {auc_s2:.3f} ({len(FEAT_S2)} features)')
print(f'   RF       Step 3A : {acc_age*100:.1f}% — âge')
print(f'   RF       Step 3B : {acc_gen*100:.1f}% — genre AUC {auc_gen:.3f}')
print(f'   RF       Step 4  : {acc_temp*100:.1f}% — temporel')


## 💾 10 — Sauvegarde

In [ ]:
joblib.dump(rf_crime,     'rf_crime_group.pkl')
joblib.dump(xgb_sev,      'xgb_severity.pkl')
joblib.dump(rf_temporal,  'rf_temporal.pkl')
joblib.dump(rf_age,       'rf_profiling_age.pkl')
joblib.dump(rf_gender,    'rf_profiling_gender.pkl')
joblib.dump(kmeans,       'kmeans_nyc.pkl')
joblib.dump(scaler_kmeans,'scaler_kmeans.pkl')
joblib.dump(le_group,     'le_crime_group.pkl')
joblib.dump(le_sev,       'le_severity.pkl')
joblib.dump(le_age,       'le_profiling_age.pkl')

print('✅ Modèles sauvegardés :')
print('   rf_crime_group.pkl      — Step 1 : type de crime')
print('   xgb_severity.pkl        — Step 2 : sévérité')
print('   rf_temporal.pkl         — Step 4 : prédiction temporelle')
print('   rf_profiling_age.pkl    — Step 3A : âge')
print('   rf_profiling_gender.pkl — Step 3B : genre')
print('   kmeans_nyc.pkl          — K-Means boroughs')
print()
print('📊 Visualisations :')
for f in sorted(os.listdir('.')):
    if f.startswith('viz_') and f.endswith('.png'):
        print(f'   {f}')
